# Ingest Circuits Files

## Purpose
This notebook demonstrates how to ingest the circuits.csv file from the landing volume into the bronze layer.

## What this notebook covers
* Read the raw CSV file
* Compare different CSV read options such as header, inferSchema, and read modes
* Apply an explicit schema for consistent ingestion
* Add metadata columns such as source file and ingestion timestamp
* Write the final result to the bronze table formula1.bronze.circuits

## Expected outcome
A Delta table in the bronze layer containing circuit reference data with ingestion metadata for downstream use.

In [0]:
spark.read.csv("/Volumes/formula1/landing/files/circuits.csv")

In [0]:
circuits_df=spark.read.csv("/Volumes/formula1/landing/files/circuits.csv")
circuits_df.show()

In [0]:
circuits_df=spark.read.format("csv").load("/Volumes/formula1/landing/files/circuits.csv")
display(circuits_df)

In [0]:
circuits_df=spark.read.format("csv").option("header","true").load("/Volumes/formula1/landing/files/circuits.csv")
display(circuits_df)

In [0]:
circuits_df=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/formula1/landing/files/circuits.csv")
display(circuits_df)

In [0]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType,DoubleType
circuits_schema=StructType([StructField("circuitId",IntegerType()),
                                  StructField("url",StringType()),
                                  StructField("circuitName",StringType()),
                                  StructField("lat",DoubleType()),
                                  StructField("long",DoubleType()),
                                  StructField("locality",StringType()),
                                  StructField("country",StringType())
                                 ])
circuits_df=spark.read.format("csv").option("header","true").schema(circuits_schema).load("/Volumes/formula1/landing/files/circuits.csv")
display(circuits_df)

## Step 2: Add Metadata Columns
In this step, the dataset is enriched with operational metadata for traceability.

* source_file captures the file path of the input record
* ingestion_timestamp captures when the data was loaded

In [0]:
from pyspark.sql import functions as F
circuits_final_df=(circuits_df.withColumn('ingestion_timestamp',F.current_timestamp()))
display(circuits_final_df)

In [0]:
from pyspark.sql import functions as F
circuits_final_df=(circuits_df.withColumn('ingestion_timestamp',F.current_timestamp())
                              .withColumn('source_file',F.col('_metadata.file_path')))
display(circuits_final_df)

## Step 3: Write to Bronze Delta Table

In [0]:
(
    circuits_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("formula1.bronze.circuits")
)

In [0]:
%sql
select * from formula1.bronze.circuits

In [0]:
display(spark.table('formula1.bronze.circuits'))

In [0]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType,DoubleType
circuits_schema=StructType([StructField("circuitId",IntegerType()),
                                  StructField("url",StringType()),
                                  StructField("circuitName",StringType()),
                                  StructField("lat",DoubleType()),
                                  StructField("long",DoubleType()),
                                  StructField("locality",StringType()),
                                  StructField("country",StringType())
                                 ])
circuits_df=spark.read.format("csv").option("header","true").option("mode","PERMISSIVE").schema(circuits_schema).load("/Volumes/formula1/landing/files/circuits.csv")
display(circuits_df)

In [0]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType,DoubleType
circuits_schema=StructType([StructField("circuitId",IntegerType()),
                                  StructField("url",StringType()),
                                  StructField("circuitName",StringType()),
                                  StructField("lat",DoubleType()),
                                  StructField("long",DoubleType()),
                                  StructField("locality",StringType()),
                                  StructField("country",DoubleType())
                                 ])
circuits_df=spark.read.format("csv").option("header","true").option("mode","PERMISSIVE").schema(circuits_schema).load("/Volumes/formula1/landing/files/circuits.csv")
display(circuits_df)

In [0]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType,DoubleType
circuits_schema=StructType([StructField("circuitId",IntegerType()),
                                  StructField("url",StringType()),
                                  StructField("circuitName",StringType()),
                                  StructField("lat",DoubleType()),
                                  StructField("long",DoubleType()),
                                  StructField("locality",StringType()),
                                  StructField("country",StringType())
                                 ])
circuits_df=spark.read.format("csv").option("header","true").option("mode","FAILFAST").load("dbfs:/Volumes/formula1/landing/files/circuits.csv")
display(circuits_df)

In [0]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType,DoubleType
circuits_schema=StructType([StructField("circuitId",IntegerType()),
                                  StructField("url",StringType()),
                                  StructField("circuitName",StringType()),
                                  StructField("lat",DoubleType()),
                                  StructField("long",DoubleType()),
                                  StructField("locality",StringType()),
                                  StructField("country",StringType())
                                 ])
circuits_df=spark.read.format("csv").option("header","true").option("mode","DROPMALFORMED").schema(circuits_schema).load("/Volumes/formula1/landing/files/circuits.csv")
display(circuits_df)

In [0]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType,DoubleType
circuits_schema=StructType([StructField("circuitId",IntegerType()),
                                  StructField("url",StringType()),
                                  StructField("circuitName",StringType()),
                                  StructField("lat",DoubleType()),
                                  StructField("long",DoubleType()),
                                  StructField("locality",StringType()),
                                  StructField("country",DoubleType())
                                 ])
circuits_df=spark.read.format("csv").option("header","true").option("mode","DROPMALFORMED").schema(circuits_schema).load("dbfs:/Volumes/formula1/landing/files/circuits.csv")
display(circuits_df)